# Step 11 on Kaggle: label the Olist reviews with a local LLM on free T4 GPUs

Same code as on your laptop (`python -m ecom_etl enrich-reviews`), pointed at an Ollama server running
inside this notebook. Nothing leaves Kaggle except the output files you download.

Before running: **Settings → Accelerator → GPU T4 x2**, **Internet → on** (needs a phone-verified account),
and **Add Input → Datasets → olistbr/brazilian-ecommerce**. Then *Run all*. About 10 minutes of setup;
the labelling time is printed at the end and goes into `docs/RESULTS.md`.

In [ ]:
import glob, os, subprocess, time
REVIEWS = glob.glob('/kaggle/input/**/olist_order_reviews_dataset.csv', recursive=True)
assert REVIEWS, 'Add the dataset olistbr/brazilian-ecommerce as an input first'
REVIEWS = REVIEWS[0]
MODEL = 'qwen2.5:7b-instruct'   # 4.7 GB: fits one T4. Try qwen2.5:14b-instruct for better topics, at ~half the speed
OUT = '/kaggle/working/enrich_olist'
os.environ['ECOM_ETL_REPORTS_DIR'] = OUT   # the report goes next to the data
print(REVIEWS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Ollama: the installer needs zstd; NUM_PARALLEL lets the GPU work on 4 requests at once
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
os.environ['OLLAMA_NUM_PARALLEL'] = '4'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}

In [ ]:
# the project itself, straight from GitHub (only pandas is needed for this step)
!pip -q install "git+https://github.com/manojhunasimarad/ETL.git" 

In [ ]:
# 1. what it will cost, 2. a 200-comment trial, 3. everything (the cache means the trial is not paid twice)
!python -m ecom_etl enrich-reviews --source {REVIEWS} --llm ollama --model {MODEL} --out {OUT} --dry-run
!python -m ecom_etl enrich-reviews --source {REVIEWS} --llm ollama --model {MODEL} --out {OUT} --limit 200 --max-quarantine 1

In [ ]:
!python -m ecom_etl enrich-reviews --source {REVIEWS} --llm ollama --model {MODEL} --out {OUT}

In [ ]:
# download these from the Output panel and copy them to data/enriched/olist/ and reports/ on your laptop
!ls -la {OUT}
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used --format=csv